## 📔 Notebook: Enhancing CrewAI Agents with Advanced Tools
This notebook teaches you how to supercharge your CrewAI agents with three core toolsets:

   1. **FileSearchTool (RAG)**: Extracting semantic knowledge from PDFs or text files.
   2. **Web Search Tool**: Fetching real-time external data from the web.
   3. **Final Analyst**: combines both sources and produces a comparison.

```
                 AAPL ANALYSIS
                      │
          ┌───────────┴───────────┐
          │                       │
          ▼                       ▼
   INTERNAL DOCUMENT          CURRENT WEB
          │                       │
          ▼                       ▼
  Internal Analyst          Market Researcher
          │                       │
    PDFSearchTool             Web Search
          │                       │
          └───────────┬───────────┘
                      ▼
              Senior Analyst
                      │
                      ▼
              Combined Analysis

------------------------------
## 🛠️ Step 1: Dependencies Installation & Environment Setup
Run the cell below to pin the exact versions of the libraries you are using.


In [ ]:
!pip install langchain==1.3.6 langchain-community==0.4.2 langchain-openai==1.2.2 langchain-text-splitters==1.1.2 pypdf==6.10.2 pinecone==7.3.0 crewai==1.15.21 crewai-tools==1.15.21 -q


In [12]:
# Check package version

import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
           "langchain-text-splitters", "pypdf", "pinecone",
           "crewai", "crewai-tools"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-text-splitters version: 1.1.2
pypdf version: 6.10.2
pinecone version: 7.3.0
crewai version: 1.15.21
crewai-tools version: 1.15.21


# Environment Configuration


Next, configure your API keys. We will use OpenAI for the LLM core and Serper for web searches (a fast, reliable Google Search API commonly integrated into CrewAI).


In [13]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

# OpenAI API Key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

# Serper API Key
if "SERPER_API_KEY" not in os.environ:
    os.environ["SERPER_API_KEY"] = getpass("Enter Serper API Key: ")

# Default LLM model
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

------------------------------
## 🔍 Step 2: Initialize the Specialized CrewAI Tools
CrewAI separates file operations into specialized RAG search tools (PDFSearchTool) and runtime environments (CodeInterpreterTool). Let's instantiate them.


In [14]:
from crewai_tools import SerperDevTool, PDFSearchTool
# from crewai.tools import CodeInterpreterTool

# 1. Web Search Tool
web_search_tool = SerperDevTool()

# 2. File Search Tool (RAG for PDF)
# Note: For text files, you can use TXTSearchTool or pass a .txt file to the generic search capabilities.
# Make sure to have a file named 'quarterly_report.pdf' in your working directory.
pdf_search_tool = PDFSearchTool(pdf='AAPL_Report.pdf')


print("✅ Tools successfully initialized!")


✅ Tools successfully initialized!


------------------------------
## 🤖 Step 3: Define the Multi-Agent Team
We will create a three-agent crew where each agent specializes in using one of the primary tools we configured above.


In [15]:
# Defining Specialized Agents
from crewai import Agent

# Agent 1: The Researcher (Uses the RAG File Search Tool)
internal_analyst = Agent(
    role="Internal AAPL Research Analyst",

    goal=(
        "Extract financial data, historical metrics, internal analyst "
        "notes, risks, and investment assumptions from the internal "
        "AAPL research document."
    ),

    backstory="""
    You are an experienced financial analyst specializing in
    internal company research documents. You carefully extract
    facts from proprietary reports and never assume that internal
    information represents the current market situation.
    """,

    tools=[pdf_search_tool],

    verbose=True
)


# Agent 2: Uses the Web Search Tool
market_researcher = Agent(
    role="Global Market Researcher",

    goal=(
        "Find current AAPL market information, recent financial "
        "results, stock information, company developments, and "
        "relevant external market context."
    ),

    backstory="""
    You are an experienced market researcher specializing in
    technology companies and public markets. You use current
    web information to validate and update information from
    internal research documents.
    """,

    tools=[web_search_tool],

    verbose=True
)


# Agent 3: Combine the result of the above 2 agent and checks for differences
synthesis_agent = Agent(
    role="Senior AAPL Research Analyst",

    goal=(
        "Compare internal AAPL research with current external "
        "market information and identify important changes, "
        "discrepancies, and updated insights."
    ),

    backstory="""
    You are a senior equity research analyst. You compare
    proprietary research with current public information.
    You identify outdated assumptions, changes in financial
    performance, and differences between historical internal
    analysis and current market conditions.
    """,

    verbose=True
)




------------------------------
## 📋 Step 4: Map Out Sequential Tasks
Each task passes its findings down the chain, combining the unique output of every tool.


In [16]:
# Defining Tasks

from crewai import Task

# Task 1: Semantic extraction from local file
internal_task = Task(
    description="""
    Analyze the internal AAPL research PDF.

    Extract:

    1. Historical revenue
    2. Net income
    3. EPS
    4. Historical stock price information
    5. Internal valuation assumptions
    6. Internal analyst view of Apple's strengths
    7. Internal analyst view of Apple's risks
    8. Internal assumptions about Apple's AI strategy

    Use only the internal PDF.
    """,

    expected_output="""
    A structured summary of the internal AAPL research,
    including financial metrics, valuation assumptions,
    strengths, risks, and AI strategy.
    """,

    agent=internal_analyst
)


# Task 2: Comparative web search
market_task = Task(
    description="""
    Research the current state of Apple (AAPL) on the web.

    Find:

    1. Most recent quarterly revenue
    2. Most recent EPS
    3. Current AAPL stock information
    4. Recent iPhone performance
    5. Recent Services performance
    6. Current Apple AI developments
    7. Recent management/company developments
    8. Important current risks or market issues

    Prioritize recent and reliable sources.
    Clearly distinguish current information from historical information.
    """,

    expected_output="""
    A structured current-market report covering:
    financial performance, stock information, products,
    Services, AI developments, management developments,
    and current risks.
    """,

    agent=market_researcher
)

In [17]:

synthesis_task = Task(
    description="""
    Compare the internal AAPL research with the current
    external market research.

    Identify:

    1. Which internal financial figures have changed?
    2. Which internal assumptions are now outdated?
    3. What current developments were not present in the
       internal report?
    4. Are there significant differences between the internal
       research and current external information?
    5. What are the major current opportunities and risks?
    6. Produce a concise executive briefing.

    Do not blindly accept either source.
    Clearly identify whether information comes from the
    internal PDF or current external research.
    """,

    expected_output="""
    An executive AAPL research briefing containing:

    - Internal findings
    - Current market findings
    - Important changes
    - Outdated assumptions
    - Current opportunities
    - Current risks
    - Source attribution
    """,

    agent=synthesis_agent,

    context=[internal_task, market_task]
)


------------------------------
## 🚀 Step 5: Assemble and Kick Off the Crew
We will now bind the agents and tasks into a single collaborative Crew.


In [18]:
# Running the Crew

from crewai import Crew, Process

# Assemble the crew
analysis_crew = Crew(
    agents=[ internal_analyst, market_researcher, synthesis_agent ],
    tasks =[ internal_task,    market_task,       synthesis_task  ],

    process=Process.sequential,
    verbose=True
)


In [20]:
# Execute the workflow
print("🚀 Starting the Crew execution loop...")

# final_report = analysis_crew.kickoff()

final_report = await analysis_crew.kickoff_async()

print("\n================ FINAL REPORT ================\n")
print(final_report)



🚀 Starting the Crew execution loop...


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.21                                                                                       │
│  Latest version:  1.15.22                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5ede4d11-820f-4c52-9a8f-f28ab80b224a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analyze the internal AAPL research PDF.                                                                    │
│                                                                                                                 │
│      Extract:                                                                                                   │
│                                                                                                                 │
│      1. Historical revenue                                                                                      │
│      2. Net income                                                                                              │
│      3. EPS                                                                                                     │
│      4. Historical stock price information                                                                      │
│      5. Internal valuation assumptions                                                                          │
│      6. Internal analyst view of Apple's strengths                                                              │
│      7. Internal analyst view of Apple's risks                                                                  │
│      8. Internal assumptions about Apple's AI strategy                                                          │
│                                                                                                                 │
│      Use only the internal PDF.                                                                                 │
│                                                                                                                 │
│  ID: dbf1376c-6bfe-4d5b-8c5f-23c0aa30bf78                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internal AAPL Research Analyst                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analyze the internal AAPL research PDF.                                                                    │
│                                                                                                                 │
│      Extract:                                                                                                   │
│                                                                                                                 │
│      1. Historical revenue                                                                                      │
│      2. Net income                                                                                              │
│      3. EPS                                                                                                     │
│      4. Historical stock price information                                                                      │
│      5. Internal valuation assumptions                                                                          │
│      6. Internal analyst view of Apple's strengths                                                              │
│      7. Internal analyst view of Apple's risks                                                                  │
│      8. Internal assumptions about Apple's AI strategy                                                          │
│                                                                                                                 │
│      Use only the internal PDF.                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior AAPL Research Analyst                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Compare the internal AAPL research with the current                                                        │
│      external market research.                                                                                  │
│                                                                                                                 │
│      Identify:                                                                                                  │
│                                                                                                                 │
│      1. Which internal financial figures have changed?                                                          │
│      2. Which internal assumptions are now outdated?                                                            │
│      3. What current developments were not present in the                                                       │
│         internal report?                                                                                        │
│      4. Are there significant differences between the internal                                                  │
│         research and current external information?                                                              │
│      5. What are the major current opportunities and risks?                                                     │
│      6. Produce a concise executive briefing.                                                                   │
│                                                                                                                 │
│      Do not blindly accept either source.                                                                       │
│      Clearly identify whether information comes from the                                                        │
│      internal PDF or current external research.                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior AAPL Research Analyst                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Executive AAPL Research Briefing                                                                           │
│                                                                                                                 │
│  #### 1. Internal Findings                                                                                      │
│  - **Financial Metrics**:                                                                                       │
│    - FY2023 Revenue: Internal research shows **$383.3 billion**; however, Q4 2023 reported revenue is **$89.5   │
│  billion** for the quarter, reflecting a 1% year-over-year decrease.                                            │
│    - FY2023 EPS: Internal research reports **$6.13**; current diluted EPS for Q4 2023 is **$1.46**, up 13%      │
│  year-over-year.                                                                                                │
│    - Historical Stock Price: Internal reference price from October 2024 is **$225.91**; current trading price   │
│  as of latest update is **$336.13**.                                                                            │
│                                                                                                                 │
│  - **Valuation Assumptions**:                                                                                   │
│    - The internal blended target price of **$255** appears outdated considering the current stock price of      │
│  **$336.13**.                                                                                                   │
│                                                                                                                 │
│  #### 2. Current Market Findings                                                                                │
│  - Current external reports indicate:                                                                           │
│    - **iPhone Revenue** for Q4 2023 stands at **$43.81 billion**, aligning with internal assumptions of high    │
│  iPhone contribution but raising concerns about over-dependence on this segment.                                │
│    - **Services Revenue** surged to **$22.3 billion** in Q4 2023, outpacing the internal projections.           │
│    - The market capitalization is more reflective of the valuation shifts with current trading at **$336.13**.  │
│                                                                                                                 │
│  #### 3. Important Changes                                                                                      │
│  - The **Q4 revenue performance** indicates a divergence from the expected growth anticipated in FY2024.        │
│  - **Services growth** has accelerated beyond the previously expected 13%, reflecting current market dynamics   │
│  and demand.                                                                                                    │
│  - The **AI strategy** has evolved, with recent acquisitions aiming to enhance Apple's capabilities in the AI   │
│  space—a significant development not accounted for in the internal report.                                      │
│                                                                                                                 │
│  #### 4. Outdated Assumptions                          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Compare the internal AAPL research with the current                                                        │
│      external market research.                                                                                  │
│                                                                                                                 │
│      Identify:                                                                                                  │
│                                                                                                                 │
│      1. Which internal financial figures have changed?                                                          │
│      2. Which internal assumptions are now outdated?                                                            │
│      3. What current developments were not present in the                                                       │
│         internal report?                                                                                        │
│      4. Are there significant differences between the internal                                                  │
│         research and current external information?                                                              │
│      5. What are the major current opportunities and risks?                                                     │
│      6. Produce a concise executive briefing.                                                                   │
│                                                                                                                 │
│      Do not blindly accept either source.                                                                       │
│      Clearly identify whether information comes from the                                                        │
│      internal PDF or current external research.                                                                 │
│                                                                                                                 │
│  Agent: Senior AAPL Research Analyst                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5ede4d11-820f-4c52-9a8f-f28ab80b224a                                                                       │
│  Final Output: ### Executive AAPL Research Briefing                                                             │
│                                                                                                                 │
│  #### 1. Internal Findings                                                                                      │
│  - **Financial Metrics**:                                                                                       │
│    - FY2023 Revenue: Internal research shows **$383.3 billion**; however, Q4 2023 reported revenue is **$89.5   │
│  billion** for the quarter, reflecting a 1% year-over-year decrease.                                            │
│    - FY2023 EPS: Internal research reports **$6.13**; current diluted EPS for Q4 2023 is **$1.46**, up 13%      │
│  year-over-year.                                                                                                │
│    - Historical Stock Price: Internal reference price from October 2024 is **$225.91**; current trading price   │
│  as of latest update is **$336.13**.                                                                            │
│                                                                                                                 │
│  - **Valuation Assumptions**:                                                                                   │
│    - The internal blended target price of **$255** appears outdated considering the current stock price of      │
│  **$336.13**.                                                                                                   │
│                                                                                                                 │
│  #### 2. Current Market Findings                                                                                │
│  - Current external reports indicate:                                                                           │
│    - **iPhone Revenue** for Q4 2023 stands at **$43.81 billion**, aligning with internal assumptions of high    │
│  iPhone contribution but raising concerns about over-dependence on this segment.                                │
│    - **Services Revenue** surged to **$22.3 billion** in Q4 2023, outpacing the internal projections.           │
│    - The market capitalization is more reflective of the valuation shifts with current trading at **$336.13**.  │
│                                                                                                                 │
│  #### 3. Important Changes                                                                                      │
│  - The **Q4 revenue performance** indicates a divergence from the expected growth anticipated in FY2024.        │
│  - **Services growth** has accelerated beyond the previously expected 13%, reflecting current market dynamics   │
│  and demand.                                                                                                    │
│  - The **AI strategy** has evolved, with recent acquisitions aiming to enhance Apple's capabilities in the AI   │
│  space—a significant development not accounted for in the internal report.                                      │
│                                                                                                                 │
│  #### 4. Outdated Assumptions                         